# 03 - Train and evaluate the ensemble step by step

This notebook intentionally performs the complete training workflow instead of loading the saved model. Change the hyperparameters, rerun the cells, inspect the DataFrames, and compare results.

**Important:** this notebook uses one row per track from the current `features.csv`. Five-second sliding-window training/inference is intentionally postponed to a later step.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
FEATURES_PATH = ROOT / 'data' / 'processed' / 'features.csv'
MODEL_PATH = ROOT / 'models' / 'music_detector_pipeline.joblib'
METRICS_PATH = ROOT / 'models' / 'training_metrics.json'
RANDOM_STATE = 42
features = pd.read_csv(FEATURES_PATH)
print(f'Loaded {features.shape[0]} tracks and {features.shape[1] - 2} features.')

## 2. Inspect the input table

The first two columns identify each track. Every remaining column is a numeric audio feature.

In [ ]:
display(features.head())
display(features['label'].value_counts().rename_axis('label').to_frame('track_count'))
print('Missing values:', int(features.isna().sum().sum()))
print('Duplicate track IDs:', int(features['track_id'].duplicated().sum()))

## 3. Separate features and labels

The model receives only the 184 numeric feature columns. Labels are encoded as `0 = human` and `1 = ai_generated`.

In [ ]:
feature_columns = [column for column in features.columns if column not in {'track_id', 'label'}]
X = features[feature_columns].to_numpy(dtype=np.float64)
y = (features['label'] == 'ai_generated').astype(np.int8).to_numpy()
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Class values:', np.unique(y, return_counts=True))

## 4. Create a stratified train/test split

The test set is held out until evaluation. `stratify=y` keeps both classes balanced in both partitions. The fixed random seed makes the experiment reproducible.

In [ ]:
TEST_SIZE = 0.20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train rows: {len(y_train)}')
print(f'Test rows: {len(y_test)}')
display(pd.DataFrame({'train': pd.Series(y_train).value_counts(), 'test': pd.Series(y_test).value_counts()}).fillna(0).astype(int))

## 5. Tune the hyperparameters here

Edit these values and rerun from this cell onward. Scaling is kept inside each pipeline, so it is fitted only on training data and automatically reused for test data.

In [ ]:
XGB_PARAMS = {
    'n_estimators': 200,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
}
SVM_PARAMS = {'C': 2.0, 'kernel': 'rbf', 'probability': True, 'class_weight': 'balanced'}
RF_PARAMS = {'n_estimators': 300, 'class_weight': 'balanced'}
VOTE_WEIGHTS = [2, 1, 1]
display(pd.DataFrame({'model': ['XGBoost', 'SVM', 'Random Forest'], 'parameters': [XGB_PARAMS, SVM_PARAMS, RF_PARAMS]}))

## 6. Build the three model pipelines

The ensemble uses soft voting: each model contributes a probability, and the weighted average becomes the final AI-generated probability.

In [ ]:
xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(**XGB_PARAMS, objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)),
])
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(**SVM_PARAMS, random_state=RANDOM_STATE)),
])
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(**RF_PARAMS, random_state=RANDOM_STATE, n_jobs=-1)),
])
ensemble = VotingClassifier(
    estimators=[('xgboost', xgb_pipeline), ('svm', svm_pipeline), ('random_forest', rf_pipeline)],
    voting='soft', weights=VOTE_WEIGHTS, n_jobs=-1
)
display(ensemble)

## 7. Train each model separately for comparison

In [ ]:
individual_models = {'XGBoost': xgb_pipeline, 'SVM': svm_pipeline, 'Random Forest': rf_pipeline}
individual_results = []
for name, model in individual_models.items():
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(np.int8)
    individual_results.append({'model': name, 'accuracy': accuracy_score(y_test, predictions), 'roc_auc': roc_auc_score(y_test, probabilities)})
display(pd.DataFrame(individual_results).sort_values('roc_auc', ascending=False))

## 8. Train the soft-voting ensemble

In [ ]:
ensemble.fit(X_train, y_train)
test_probabilities = ensemble.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= 0.5).astype(np.int8)
metrics = {
    'test_accuracy': accuracy_score(y_test, test_predictions),
    'test_roc_auc': roc_auc_score(y_test, test_probabilities),
}
display(pd.DataFrame([metrics]))
print(classification_report(y_test, test_predictions, target_names=['human', 'ai_generated']))

## 9. Inspect errors and confidence

In [ ]:
results = pd.DataFrame({'actual': y_test, 'ai_probability': test_probabilities, 'prediction': test_predictions})
results['actual_label'] = results['actual'].map({0: 'human', 1: 'ai_generated'})
results['predicted_label'] = results['prediction'].map({0: 'human', 1: 'ai_generated'})
results['correct'] = results['actual'] == results['prediction']
display(results.sort_values('ai_probability', ascending=False).head(20))
display(results[~results['correct']])

## 10. Plot evaluation results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, test_predictions), display_labels=['human', 'ai_generated']).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Held-out confusion matrix')
RocCurveDisplay.from_predictions(y_test, test_probabilities, ax=axes[1])
axes[1].set_title(f"Held-out ROC curve (AUC={metrics['test_roc_auc']:.3f})")
plt.tight_layout()
plt.show()

## 11. Inspect XGBoost feature importance

This is one model's importance view, not a causal explanation. It is useful for experimentation and for deciding which feature groups to investigate.

In [ ]:
xgb_classifier = ensemble.named_estimators_['xgboost'].named_steps['classifier']
importance = pd.DataFrame({'feature': feature_columns, 'importance': xgb_classifier.feature_importances_}).sort_values('importance', ascending=False)
display(importance.head(25))
importance.head(20).sort_values('importance').plot.barh(x='feature', y='importance', figsize=(10, 7), legend=False)
plt.title('Top XGBoost feature importances')
plt.show()

## 12. Save the notebook-trained model (optional)

This saves the same artifact shape used by the command-line pipeline. Run this only after you are satisfied with the experiment.

In [ ]:
save_artifact = True
if save_artifact:
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump({'model': ensemble, 'feature_columns': feature_columns, 'label_mapping': {'human': 0, 'ai_generated': 1}}, MODEL_PATH)
    METRICS_PATH.write_text(json.dumps({**metrics, 'feature_count': len(feature_columns), 'random_state': RANDOM_STATE}, indent=2), encoding='utf-8')
    print(f'Saved model to {MODEL_PATH}')
    print(f'Saved metrics to {METRICS_PATH}')